# Capítulo 15 — Producción y cierre

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** El curso termina, pero Nova Assist y el equipo de investigación tendrían que seguir funcionando mañana… o apagarse sin dejar recursos facturando.

| | |
|---|---|
| **🧠 Componente agéntico** | Ciclo de vida y operación |
| **La necesidad** | Desplegar con garantías y dejar la cuenta limpia |
| **Lo que construimos** | Entornos, *pipeline* con *quality gates*, versiones y limpieza |
| **🏠 En local** | ☁️ No aplica: operar es mantener un sistema desplegado. |
| **☁️ Por qué AWS al salir a Internet** | Los scripts manuales no son repetibles ni auditables. La CLI despliega con CDK, la evaluación por lotes actúa como *quality gate* y los recursos fuera de la CLI se inventarían y se borran. |
| **En el ordenador sería…** | El **mantenimiento**: actualizar con garantías, vigilar y apagar lo que no se usa |
| **Requisitos del caso** | RNF-02, operación |
| **Prerrequisito · duración** | Capítulos anteriores · 30 min |
| **⚠️ Dónde se rompe la analogía** | El mantenimiento de un ordenador es de una máquina; aquí es de un sistema **distribuido entre servicios**: lo que no está en la CLI hay que inventariarlo a mano. |

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(15)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(15)
# ponerse_al_dia(15)   # ← descomenta si llegas tarde o algo se rompió

## 🧠 La teoría: ciclo de vida

- Entornos, versiones, despliegue con *quality gates*, vigilancia, *rollback* y retirada.
- Un sistema agéntico es **distribuido**: lo que no está declarado como código hay que inventariarlo a mano.

> Este componente **no existe en local**: operar es mantener un sistema desplegado.

## 0. El balance del viaje

In [ ]:
cuaderno()
taximetro_historico()

## 1. Hacia producción

- **Entornos:** declara cuentas y regiones de dev, pre y prod en `agentcore/aws-targets.json`. La CLI despliega con **CDK**, y `agentcore/cdk/` se puede versionar e integrar en el *pipeline*.
- ***Pipeline* recomendado:**
  1. `agentcore validate`;
  2. tests de la Lambda;
  3. `agentcore deploy` en pre;
  4. **batch evaluation como *quality gate***;
  5. despliegue en prod;
  6. A/B si el cambio afecta al comportamiento;
  7. promoción.
- **Versiones y *endpoints*** del Runtime para promocionar y hacer *rollback*.
- **Antes del primer despliegue real**, repasa el checklist de la sección 16 del documento.

## 2. Limpieza

> ⚠️ Las celdas siguientes **eliminan recursos**. Solo se ejecutan si pones `CONFIRMAR_LIMPIEZA = True`.

In [ ]:
CONFIRMAR_LIMPIEZA = False   # ← cámbialo a True de forma consciente


def intentar(descripcion, fn):
    """Ejecuta una operación de borrado y continúa aunque falle (recurso ya borrado, sin permisos…)."""
    try:
        fn()
        print("✔", descripcion)
    except Exception as e:  # noqa: BLE001
        print("·", descripcion, "→", type(e).__name__, e)

### 2.1 Recursos gestionados por la CLI (runtime, memoria, gateway, políticas, evaluaciones…)

In [ ]:
assert CONFIRMAR_LIMPIEZA, "Limpieza no confirmada"
!cd "{PROJECT}" && agentcore remove all && agentcore deploy --yes

### 2.2 Recursos creados fuera de la CLI

In [ ]:
assert CONFIRMAR_LIMPIEZA, "Limpieza no confirmada"
import boto3

s = state_load()
r = region()

def intentar(descripcion, fn):
    try:
        fn()
        print("✔", descripcion)
    except Exception as e:  # noqa: BLE001
        print("·", descripcion, "→", type(e).__name__, e)

lam, iam = boto3.client("lambda", region_name=r), boto3.client("iam")
idp, s3 = boto3.client("cognito-idp", region_name=r), boto3.resource("s3", region_name=r)
bedrock = boto3.client("bedrock", region_name=r)

for funcion, rol, inline in (("nova-pedidos", "nova-pedidos-role", "LecturaIcebergAthena"),
                             ("nova-politicas", "nova-politicas-role", "BusquedaVectorial")):
    intentar(f"Lambda {funcion}", lambda f=funcion: lam.delete_function(FunctionName=f))
    intentar(f"Política en línea de {rol}", lambda r_=rol, i=inline: iam.delete_role_policy(RoleName=r_, PolicyName=i))
    intentar(f"Política gestionada de {rol}", lambda r_=rol: iam.detach_role_policy(
        RoleName=r_, PolicyArn="arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole"))
    intentar(f"Rol {rol}", lambda r_=rol: iam.delete_role(RoleName=r_))

# S3 Vectors: índice y vector bucket
s3v = boto3.client("s3vectors", region_name=r)
if s.get("VECTOR_BUCKET"):
    if s.get("VECTOR_INDEX"):
        intentar("Índice vectorial", lambda: s3v.delete_index(vectorBucketName=s["VECTOR_BUCKET"], indexName=s["VECTOR_INDEX"]))
    intentar("Vector bucket", lambda: s3v.delete_vector_bucket(vectorBucketName=s["VECTOR_BUCKET"]))

# S3 Tables: tablas, namespace y table bucket
s3t = boto3.client("s3tables", region_name=r)
if s.get("TABLE_BUCKET_ARN"):
    tb = s["TABLE_BUCKET_ARN"]
    for t in s3t.list_tables(tableBucketARN=tb).get("tables", []):
        intentar(f"Tabla {t['name']}", lambda t=t: s3t.delete_table(tableBucketARN=tb, namespace=t["namespace"][0], name=t["name"]))
    intentar("Namespace tiendanova", lambda: s3t.delete_namespace(tableBucketARN=tb, namespace="tiendanova"))
    intentar("Table bucket", lambda: s3t.delete_table_bucket(tableBucketARN=tb))
if s.get("M2M_POOL_ID"):
    intentar("Dominio Cognito M2M", lambda: idp.delete_user_pool_domain(Domain=s["M2M_DOMAIN"], UserPoolId=s["M2M_POOL_ID"]))
    intentar("Pool M2M", lambda: idp.delete_user_pool(UserPoolId=s["M2M_POOL_ID"]))
if s.get("CLIENTES_POOL_ID"):
    intentar("Pool de clientes", lambda: idp.delete_user_pool(UserPoolId=s["CLIENTES_POOL_ID"]))
if s.get("GUARDRAIL_ID"):
    intentar("Guardrail", lambda: bedrock.delete_guardrail(guardrailIdentifier=s["GUARDRAIL_ID"]))
if s.get("LAB_BUCKET"):
    def _vaciar_y_borrar():
        b = s3.Bucket(s["LAB_BUCKET"])
        b.objects.all().delete()
        b.delete()
    intentar("Bucket del laboratorio", _vaciar_y_borrar)

### 2.3 Recursos de cuenta del capítulo 0 (solo administrador, opcional)

Solo si la cuenta deja de usarse para formación. Transaction Search y el *bootstrap* de CDK pueden quedarse: no generan coste relevante sin uso.

In [ ]:
CONFIRMAR_LIMPIEZA_CUENTA = False   # ← solo el administrador, de forma consciente
assert CONFIRMAR_LIMPIEZA_CUENTA, "Limpieza de cuenta no confirmada"
import boto3

s = state_load()
if s.get("BUDGET_NAME"):
    intentar("Presupuesto", lambda: boto3.client("budgets", region_name="us-east-1").delete_budget(
        AccountId=s["ACCOUNT_ID"], BudgetName=s["BUDGET_NAME"]))
if s.get("PARTICIPANT_ROLE_ARN"):
    iam = boto3.client("iam")
    rol = s["PARTICIPANT_ROLE_ARN"].split("/")[-1]
    for p in iam.list_attached_role_policies(RoleName=rol)["AttachedPolicies"]:
        intentar(f"Desvincular {p['PolicyName']}", lambda p=p: iam.detach_role_policy(RoleName=rol, PolicyArn=p["PolicyArn"]))
    intentar("Política en línea", lambda: iam.delete_role_policy(RoleName=rol, PolicyName="NovaLabExtras"))
    intentar("Rol del participante", lambda: iam.delete_role(RoleName=rol))
# El catálogo federado s3tablescatalog es de toda la cuenta y región: bórralo solo si nadie más usa S3 Tables
BORRAR_CATALOGO_S3TABLES = False
if BORRAR_CATALOGO_S3TABLES:
    intentar("Catálogo s3tablescatalog", lambda: boto3.client("glue", region_name=s["REGION"]).delete_catalog(
        CatalogId="s3tablescatalog"))
print("CDKToolkit y Transaction Search se conservan (elimínalos manualmente si lo necesitas).")

### 2.4 Estado local

In [ ]:
assert CONFIRMAR_LIMPIEZA, "Limpieza no confirmada"
STATE_FILE.rename(STATE_FILE.with_suffix(".json.bak"))
print("lab_state.json archivado como lab_state.json.bak")

## 3. Retos para seguir aprendiendo

1. Migra Nova Assist al **harness gestionado** (modelo, prompt y herramientas del Gateway) y compara el esfuerzo con la versión en código.
2. Añade un ***lifecycle hook*** que bloquee `procesar_reembolso` fuera del horario laboral.
3. Implementa el **derecho de supresión**: una herramienta interna que borre los registros de memoria de un `actor_id`.
4. Sustituye la Lambda por un **servidor MCP** desplegado en Runtime y usa **elicitation** para confirmar los reembolsos.
5. Crea **application inference profiles** etiquetados por canal (web o app) y construye un panel de coste por conversación.

🎓 **Fin del laboratorio.**